# PA2 stage 02: DPO HH-RLHF (Task 4)

15k single-turn Anthropic HH-RLHF pairs (helpful/harmless), validated on 1k held-out pairs. From-scratch DPO (`scripts/train_dpo.py`): policy on `cuda:0`, frozen reference on
`cuda:1`, both from the SFT `best` checkpoint. Resumes from `outputs/dpo_hh/resume` when a
previous run's output is attached. After training, the DPO and SFT best checkpoints answer
`eval_sets/hh_rlhf_eval_300_seed42.json` with identical greedy decoding (`--batch_size 32 --max_new_tokens 512 --repetition_penalty 1.1`) and the blind pair file
for DPO-vs-SFT judging is written to `outputs/gen_eval/`.

In [ ]:
REPO_URL = "https://github.com/usmanbinnaeem/small-model-alignment-starter.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log -1 --format="%h %s"

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib 
!python -m pip install -q -e . --no-deps

In [ ]:
import sys, torch, transformers, datasets
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
assert torch.cuda.device_count() == 2, "Expected GPU T4 x2"

In [ ]:
# Let PyTorch grow memory segments instead of fragmenting (inherited by every ! and subprocess call).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Link the attached prepared-data dataset to data/prepared, wherever Kaggle mounted it.
import glob
hits = glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)
assert hits, "attach the pa2-alignment-data dataset"
DATA_DIR = os.path.dirname(hits[0])
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/prepared"):
    os.symlink(DATA_DIR, "data/prepared")
!ls -la data/prepared/
!PYTHONPATH=. python -m pytest -q tests/

In [ ]:
# The SFT stage's output is attached as an input; use its best checkpoint as policy init and reference.
hits = sorted(glob.glob("/kaggle/input/**/small-model-alignment-starter/outputs/sft/best/config.json", recursive=True))
assert hits, "attach the pa2-01-sft notebook output"
SFT_BEST = os.path.dirname(hits[-1])
print("SFT best checkpoint:", SFT_BEST)
!cat {SFT_BEST}/checkpoint_info.json

In [ ]:
# Resume support: copy this stage's run folders back from a previous run's attached output.
import shutil
for run in ['dpo_hh', 'gen_eval']:
    local = os.path.join("outputs", run)
    if os.path.exists(local):
        continue
    found = sorted(glob.glob(f"/kaggle/input/**/small-model-alignment-starter/outputs/{run}", recursive=True))
    if found:
        print(f"restoring outputs/{run} from {found[-1]}")
        shutil.copytree(found[-1], local, symlinks=False)
    else:
        print(f"outputs/{run}: starting fresh")
!du -sh outputs/* 2>/dev/null || true

In [ ]:
!mkdir -p outputs/dpo_hh
!PYTHONPATH=. python -u scripts/train_dpo.py --config configs/dpo_hh.yaml --sft_checkpoint {SFT_BEST} 2>&1 | tee -a outputs/dpo_hh/train.log

In [ ]:
!PYTHONPATH=. python scripts/plot_metrics.py outputs/dpo_hh
!cat outputs/dpo_hh/summary.json

In [ ]:
# Task 6 on Kaggle: once training has finished, the DPO best and SFT best checkpoints answer this
# run's fixed evaluation set (one model per GPU, identical decoding), then the blind pair file is built.
import json
summary = json.load(open("outputs/dpo_hh/summary.json"))
assert not summary["stopped_early"], "training hit the time limit: push again with --resume to finish it first"
FLAGS = "--batch_size 32 --max_new_tokens 512 --repetition_penalty 1.1"
os.makedirs("outputs/gen_eval", exist_ok=True)
jobs = {"0": ("dpo_hh", "outputs/dpo_hh/best"), "1": ("sft", SFT_BEST)}
procs = [subprocess.Popen(
            f"python -u scripts/generate.py --model {path} --generator qwen2.5-0.5b-{name} "
            f"--eval_set eval_sets/hh_rlhf_eval_300_seed42.json --out outputs/gen_eval/hh_rlhf_eval_300_seed42__{name}.json {FLAGS}",
            shell=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONPATH": "."},
            stdout=open(f"outputs/gen_eval/gen_{name}.log", "a"), stderr=subprocess.STDOUT)
         for gpu, (name, path) in jobs.items()]
print("exit codes", [p.wait() for p in procs])
!tail -n 2 outputs/gen_eval/gen_dpo_hh.log outputs/gen_eval/gen_sft.log
!PYTHONPATH=. python scripts/build_pairs.py --a outputs/gen_eval/hh_rlhf_eval_300_seed42__sft.json --a_name sft \\
    --b outputs/gen_eval/hh_rlhf_eval_300_seed42__dpo_hh.json --b_name dpo_hh --out outputs/gen_eval/pairs_hh_rlhf_eval_300_seed42__dpo_hh_vs_sft.jsonl
for f in sorted(glob.glob("outputs/gen_eval/hh_rlhf_eval_300_seed42__*.json")):
    if f.endswith(".settings.json"):
        continue
    rows = json.load(open(f))
    n_len = sum(r["finish_reason"] == "length" for r in rows)
    print(f"{os.path.basename(f):45s} rows={len(rows)} hit_max_tokens={n_len} mean_words={sum(len(r['output'].split()) for r in rows) / len(rows):.0f}")

In [ ]:
!cd outputs && zip -qr /kaggle/working/dpo_hh_run_outputs.zip dpo_hh gen_eval -x "dpo_hh/best/*" "dpo_hh/final/*" "dpo_hh/resume/*"
!du -sh outputs/dpo_hh/* /kaggle/working/dpo_hh_run_outputs.zip